<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m7_05_regression_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M7.05 — One model, four numbers

In the lab we met several ways of scoring a regression model: the mean
absolute error, the root mean squared error, R², the percentage error,
and the plot of predicted against real values. In this workbook you
apply them to the model that **follows Parkinson's patients over
time**. You will see why the same predictions can deserve an R² of 0.8
or of −20, where a percentage error misleads, and how sure we can be of
a score computed on few patients.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "higher_r2_on": "ca978112ca1bbdca",
    "honest_range": "306155ab7f9ebe69",
    "mae_example": "4b227777d4dd1fc6",
    "percentage_error_mild_patient": "1a6562590ef19d10",
    "rmse_compared_with_mae": "a95aa96f841a1435",
    "within_one_patient": "e760bdbe0be65db9",
    "worst_group_by_mape": "e15a91e977feae7c"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. The model we evaluate

Workbook M7.02 found one use for which the voice model works: following
patients it already knows. We keep that setting. The model learns from
the first 70% of each patient's recording days and predicts
`total_updrs` (the severity of the symptoms) for the last 30%.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

recordings = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/parkinsons_telemonitoring.csv"
)
features = [
    name for name in recordings.columns if name not in ("subject", "test_time", "motor_updrs", "total_updrs")
]
progress = recordings.groupby("subject")["test_time"].rank(pct=True)
train, test = recordings[progress <= 0.7], recordings[progress > 0.7]

model = HistGradientBoostingRegressor(random_state=0).fit(train[features], train["total_updrs"])
results = pd.DataFrame(
    {
        "patient": test["subject"],
        "real": test["total_updrs"],
        "predicted": model.predict(test[features]),
    }
)
results["error"] = results["predicted"] - results["real"]
print(f"{len(train)} recordings to learn from, {len(results)} later recordings to predict")
results.head().round(1)

## 2. MAE and RMSE

- The **mean absolute error** (MAE) is the average size of the errors.
- The **root mean squared error** (RMSE) squares the errors, averages
  them, and takes the square root. Squaring makes large errors count
  much more than small ones.

✏️ A model makes five errors: 1, 1, 1, 1 and 16 points. What is its MAE?

In [ ]:
mae_example = ____
check("mae_example", mae_example)

✏️ For the same five errors, will the RMSE be `"smaller than"`, `"equal
to"` or `"larger than"` the MAE? (The squares are 1, 1, 1, 1 and 256.)

In [ ]:
rmse_compared_with_mae = ____
check("rmse_compared_with_mae", rmse_compared_with_mae)

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

example_errors = np.array([1, 1, 1, 1, 16])
mae = mean_absolute_error(results["real"], results["predicted"])
rmse = root_mean_squared_error(results["real"], results["predicted"])
print(f"example: MAE {np.abs(example_errors).mean():.2f}, RMSE {np.sqrt((example_errors**2).mean()):.2f}")
print(f"our model: MAE {mae:.2f}, RMSE {rmse:.2f} UPDRS points")

The RMSE is never smaller than the MAE, and the two are equal only when
all the errors have the same size. A RMSE far above the MAE is the sign
of a few very large errors. Which one to report depends on what hurts:
if an error of 16 points is much worse than sixteen errors of 1 point,
the RMSE says so; if all points of error cost the same, use the MAE.

## 3. Looking at the predictions

Before any other number, a picture. Each dot is one test recording.

In [ ]:
from sklearn.metrics import PredictionErrorDisplay

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
PredictionErrorDisplay.from_predictions(
    results["real"], results["predicted"], kind="actual_vs_predicted", ax=axes[0], scatter_kwargs={"s": 6, "alpha": 0.4}
)
PredictionErrorDisplay.from_predictions(
    results["real"], results["predicted"], kind="residual_vs_predicted", ax=axes[1], scatter_kwargs={"s": 6, "alpha": 0.4}
)
axes[0].set_title("real against predicted")
_ = axes[1].set_title("error against predicted")

On the left, a perfect model would put every dot on the diagonal. The
dots follow it from afar, in small clusters: each cluster is one
patient.

## 4. ➕ Going further: R² depends on who is in the test set

R² compares the model with the strategy "always predict the mean of the
**test** targets": 1 is perfect, 0 is no better than that mean, and
below 0 is worse.

In [ ]:
from sklearn.metrics import r2_score

overall_r2 = r2_score(results["real"], results["predicted"])
print(f"R² on all the test recordings: {overall_r2:.2f}")

A neurologist is not interested in all patients at once. She follows
**one** patient and wants to know whether he is getting better or worse.

🔮 **Predict first.** We compute R² separately on the test recordings
of each of the 42 patients, with the same predictions. Will most of
these 42 values be **"close to the overall value"**, **"around 0"** or
**"clearly negative"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["close to the overall value", "around 0", "clearly negative"])

In [ ]:
per_patient = results.groupby("patient").apply(
    lambda rows: pd.Series(
        {
            "MAE": rows["error"].abs().mean(),
            "R²": r2_score(rows["real"], rows["predicted"]),
            "spread of the real scores": rows["real"].std(),
        }
    ),
    include_groups=False,
)
print(f"patients with a negative R²: {(per_patient['R²'] < 0).sum()} of {len(per_patient)}")
print(f"spread of the real scores over all test recordings: {results['real'].std():.1f}")
per_patient.head(6).round(2)

✏️ Within a single patient, is the model better or worse than predicting
that patient's own mean score? Write `"better"` or `"worse"`.

In [ ]:
within_one_patient = ____
check("within_one_patient", within_one_patient)

(The few patients whose R² is not negative have a constant score in the
test period. R² is not defined there, and scikit-learn returns 0: the
model is no better for them.)

Nothing changed in the predictions or in the errors. What changed is
**what the model is compared with**.

- Across all patients the scores range from 7 to 55, so "predict the
  overall mean" is a poor strategy and the model beats it easily:
  R² = 0.8.
- Within one patient the score moves by a point or two in two months, so
  "predict this patient's mean" is almost perfect, and an error of 4
  points is far worse than that: R² strongly negative.

The model knows **who is ill and how much**. It does not see **how one
patient changes**, which was the whole purpose of the device. An R²
must always be read together with the question: variation of what,
among whom?

✏️ A model has an MAE of 4 points on two test sets. In test set A the
real values range from 0 to 100; in test set B they all lie between 48
and 52. On which test set is its R² higher? Write `"A"` or `"B"`.

In [ ]:
higher_r2_on = ____
check("higher_r2_on", higher_r2_on, hint="Where is 'always predict the mean' the weaker opponent?")

## 5. ➕ Going further: errors in percent

The **mean absolute percentage error** (MAPE) divides each error by the
real value. It is popular because it needs no unit: "we are 15% off".

✏️ The model is 4 points off for a patient whose real score is 8, and
4 points off for a patient whose real score is 50. What is the
percentage error for the **first** patient? Give a number of percent
(for example `12` for 12%).

In [ ]:
percentage_error_mild_patient = ____
check("percentage_error_mild_patient", percentage_error_mild_patient, hint="4 out of 8, in percent.")

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

severity = pd.qcut(results["real"], 3, labels=["mild third", "middle third", "severe third"])
by_severity = results.groupby(severity, observed=True).apply(
    lambda rows: pd.Series(
        {
            "MAE (points)": rows["error"].abs().mean(),
            "MAPE (%)": 100 * mean_absolute_percentage_error(rows["real"], rows["predicted"]),
        }
    ),
    include_groups=False,
)
print(f"MAPE on all test recordings: {100 * mean_absolute_percentage_error(results['real'], results['predicted']):.1f}%")
by_severity.round(1)

✏️ According to the MAPE, for which patients is the model the worst:
the `"mild third"` or the `"severe third"`?

In [ ]:
worst_group_by_mape = ____
check("worst_group_by_mape", worst_group_by_mape)

The same error in points weighs several times more, in percent, on a
patient with a low score. A percentage error is right when a relative
error is what matters (being €1000 off on a house is nothing, on a
bicycle it is everything). It is misleading when the scale has no
meaningful zero or when small values are common, and it explodes when a
real value is close to 0.

## 6. ➕ Going further: how sure are we of the MAE?

Our MAE is computed on about 1800 recordings. With other recordings it
would be a little different. The **bootstrap** estimates by how much: we
draw many new test sets from ours, with replacement (as for bagging in
workbook M6.01), compute the MAE of each, and look at the range that
contains 95% of them.

There are two ways to draw: resample the **recordings**, or resample
whole **patients** with all their recordings.

🔮 **Predict first.** Which way will give the **wider** range?
`"resampling recordings"` or `"resampling patients"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["resampling recordings", "resampling patients"])

In [ ]:
rng = np.random.default_rng(0)
absolute_errors = results["error"].abs().to_numpy()
errors_of_patient = [group.abs().to_numpy() for _, group in results.groupby("patient")["error"]]

maes_recordings = [
    absolute_errors[rng.integers(0, len(absolute_errors), len(absolute_errors))].mean() for _ in range(1000)
]
maes_patients = [
    np.concatenate([errors_of_patient[i] for i in rng.integers(0, 42, 42)]).mean() for _ in range(1000)
]
intervals = pd.DataFrame(
    {
        "resampling recordings": np.percentile(maes_recordings, [2.5, 97.5]),
        "resampling patients": np.percentile(maes_patients, [2.5, 97.5]),
    },
    index=["lower bound", "upper bound"],
).T
intervals["width"] = intervals["upper bound"] - intervals["lower bound"]
print(f"MAE: {mae:.2f}")
intervals.round(2)

✏️ Which range is the honest one, if the model is meant for patients in
general and not only for these 42? Write `"resampling recordings"` or
`"resampling patients"`.

In [ ]:
honest_range = ____
check("honest_range", honest_range)

Resampling recordings treats 1800 recordings as 1800 independent pieces
of evidence and announces a precise MAE. But the errors of one patient
resemble each other, and there are only 42 patients: the honest range is
several times wider. It is the last appearance of the idea that ran
through this module: **count the independent units, not the rows.**

🧠 **Think.**

1. The four numbers of this workbook, for the same predictions: MAE 4.0,
   RMSE 4.9, R² 0.8 overall and about −20 within a patient, MAPE 15%.
   Which one would you put in a brochure for investors? Which one in a
   report to neurologists? Why are both choices honest or dishonest?
2. You must choose one metric to tune a model that predicts delivery
   times, where being very late is much worse than being a little late.
   Which one, and why?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - MAE and RMSE: what each one measures and when they differ;
   - how the same predictions get an R² of 0.8 overall and a negative
     R² within each patient;
   - when a percentage error is a good idea and when it misleads;
   - the two bootstrap ranges and which one to believe.